# ST 554 - Project 2
Author: Max Campbell

## Part I - Creating a Class

In object-oriented programming, one of the foundational components of a script is a class. A simple definition of a class is that it is a framework from which we can create objects. For our purposes, objects are instances of a class, which we can manipulate using methods defined within the class. Let's take a closer look at a class to understand the powerful functionality that we can apply to our work in big data.

The first thing we need to do is write a python script with the class contained within! This includes defining the class itself, as well as the key attributes that we want to be able to access in an instance of the class. To keep things simple, we'll use just one attribute, a `pyspark` DataFrame. The full script for our example class with annotated code, `SparkDataCheck`, can be found in the "SparkDataCheck.py" file. Let's go ahead and import the modules we will need.

In [141]:
#Import/reload modules
import SparkDataCheck
from pyspark.sql import SparkSession
import pandas as pd
import importlib
importlib.reload(SparkDataCheck)

<module 'SparkDataCheck' from '/home/jupyter-mscampb4@ncsu.edu/Project2/SparkDataCheck.py'>

As we mentioned previously, we are using a `pyspark` DataFrame as the main piece of data, stored in an attribute of the class that is defined whenever we instantiate it. This means that we should create a local Spark session, then use it to import a comma-separated values (CSV) file as our DataFrame.

In [101]:
#Instantiate SparkDataCheck using air.csv
spark = SparkSession.builder.master('local[*]').appName('Project2').getOrCreate() #Initialize pyspark session
air = SparkDataCheck.SparkDataCheck.load_pyspark(spark, "air.csv")
air.df.show()

+---+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|21:00:00|   2.2|       1376|      80|     9.2|          948|    172|        1092|    122|        1584| 

26/03/21 09:52:38 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-mscampb4@ncsu.edu/Project2/air.csv


Let's take a closer look at the second line of code in the section above. First, we call `SparkDataCheck`, which is the module we previously imported. From there, we can call on our class `SparkDataCheck` to reference it directly. This creates a `SparkDataCheck` object that accepts one argument: the `pyspark` DataFrame we just imported. However, we can see that we called a function that looks something like `load_pyspark(SparkSession, string)`. This is a class method! Denoted by the `@classmethod` tag in our script, a class method is a method that can be run when the object is first created to directly modify any attributes. Class methods are extremely helpful when there are multiple ways to handle data that we want to support. For example, we created the DataFrame directly from a CSV file, but what if we wanted to create it using `pandas` instead?

In [135]:
#Alternatively, load in using a pandas df and initialize the class in this manner
airPd = pd.read_csv("air.csv")

#Do some data cleaning while we are here
airPd = airPd.replace([-200], [None])
airPd['Date'] = airPd.Date.astype('string')
airPd = airPd.rename(columns={'PT08.S1(CO)': 'PT08S1(CO)', 
                      'PT08.S2(NMHC)': 'PT08S2(NMHC)', 
                      'PT08.S3(NOx)': 'PT08S3(NOx)',
                      'PT08.S4(NO2)': 'PT08S4(NO2)',
                      'PT08.S5(O3)': 'PT08S5(O3)'}) #Removing the dots as they lead to errors

#Load it into pyspark
air = SparkDataCheck.SparkDataCheck.load_pandas(spark, airPd)

We instantiated the SparkDataCheck object using the same general format, but this time the class method was `load_pandas(SparkSession, pandas dataframe)`. Both methods arrived at the same destination, but used different types of underlying data structures to get there. Now that we have an instance of SparkDataCheck, we can use methods to manipulate it! A normal method is different from a class method in that it have to run when the class is first created. This means that we can call regular methods on an existing instance of our object. In `SparkDataCheck`, we've created a few methods that validate some aspects of the DataFrame, as well as a couple methods that generate summary statistics. Let's go through them one-by-one.

First, we have the `validate_numeric()` method. This method accepts a column as input, and returns our DataFrame with an appended column of Boolean values denoting whether the value in the specified column falls within a set of bounds that we can define as well. There are several different behaviors that we want to support. First, let's make sure we've defined what happens when a non-numeric column is input:

In [142]:
#Validate various columns for numeric values
air.validate_numeric("Date", lower = 0, upper = 100).show() #Returns error due to incorrect col type

Please provide a numeric column.
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|      1376|      80|     9.2|      

We've printed out a message, and then returned the initial DataFrame. Now, let's see what happens if the inputs are as expected.

In [143]:
air.validate_numeric("CO(GT)", lower = 0, upper = 100).show() #Evaluates

+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+---------------------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|passes_num_validation|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+---------------------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|                 true|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|                 true|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.75

We've appended a column with boolean values. This is what we wanted! Notice that when the value in the user-specified column is NULL, the new column also returns a NULL value. This is expected behavior for our purposes, as we don't want to evaluate whether a value falls within certain bounds if it doesn't exist, after all. Next up, what if the user only wants to evaluate against one bound?

In [145]:
air.validate_numeric("CO(GT)", lower = 1).show() #Evaluates
air.validate_numeric("CO(GT)", upper = 1.5).show() #Evaluates

+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+-----------------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|passes_validation|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+-----------------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|             true|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|             true|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.7502|             true

Looks like both versions of this method call are working as intended! By accounting for this behavior, we've created more use cases for the method, which will benefit us in further analysis. Lastly, what happens if no bounds are provided?

In [146]:
air.validate_numeric("CO(GT)").show() #Returns error due to lack of bounds

Please provide at least one bound.
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|      1376|      80|     9.2|    

This results in another "error" message, which is reasonable.

Let's also look at a similar method for string-based columns. For this method, we accept a user-specified column and a list of valid "levels", and return a boolean column appended to our DataFrame, similar to the previous method. There are two distinct behaviors in this method. The first of which occurs when a non-string-based column is input as an argument:

In [147]:
#Validate various columns for string values
validLevels = ["3/10/2004"]
air.validate_string("CO(GT)", validLevels).show() #Error due to ineligible data type

Please provide a string column.
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|      1376|      80|     9.2|       

This is similar to the behavior in the previous method, which is good because it is consistent. Now, let's look at a valid call of the method:

In [148]:
air.validate_string("Date", validLevels).show() #evaluates as expected

+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+---------------------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|passes_str_validation|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+---------------------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|                 true|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|                 true|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.75

Likewise, this is consistent with the previous method.

In [149]:
#A few more column arguments, for good measure.
air.validate_string("Time", validLevels).show() #evaluates to all false because no valid levels should occur
air.validate_string("Date", None).show() #NULL should evaluate to NULL

+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+---------------------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|passes_str_validation|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+---------------------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|                false|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|                false|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.75

The last validation method that we have is simple. The user will specify a column, and the DataFrame will return with Boolean values appended to it indicating whether a value in that column was NULL or not. This method is not picky, so there's really only one behavior defined:

In [151]:
#Test denote_null_values method, all calls should evaluate under expected behavior
air.denote_null_values("CO(GT)").show()
air.denote_null_values("NOx(GT)").show()
air.denote_null_values("NO2(GT)").show()
air.denote_null_values("T").show()

+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+--------------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08S1(CO)|NMHC(GT)|C6H6(GT)|PT08S2(NMHC)|NOx(GT)|PT08S3(NOx)|NO2(GT)|PT08S4(NO2)|PT08S5(O3)|   T|  RH|    AH|has_null_value|
+----------+---------+--------+------+----------+--------+--------+------------+-------+-----------+-------+-----------+----------+----+----+------+--------------+
|         0|3/10/2004|18:00:00|   2.6|      1360|     150|    11.9|        1046|    166|       1056|    113|       1692|      1268|13.6|48.9|0.7578|         false|
|         1|3/10/2004|19:00:00|   2.0|      1292|     112|     9.4|         955|    103|       1174|     92|       1559|       972|13.3|47.7|0.7255|         false|
|         2|3/10/2004|20:00:00|   2.2|      1402|      88|     9.0|         939|    131|       1140|    114|       1555|      1074|11.9|54.0|0.7502|         false|
|         3|3/10

Looks like this is working as expected! This method can be useful for filtering out missing values from a dataset before doing some more exhaustive analysis on it. 

Speaking of doing some more exhaustive analysis, let's take a look at the summarization methods that were created with this class. The first is `get_min_max()`, which simply accepts a user-specified column (and an optional grouping column), and returns the minimum and maximum values for each applicable group (whether it be a whole column, or those within a group defined by the grouping column). The benefits of creating this method include being able to seamlessly handle a grouping column without asking the user to augment their program significantly. Note that the remaining methods all return `pandas` dataframes. Let's check out some examples!

In [152]:
#Test get_min_max
air.get_min_max(col = "CO(GT)") #Returns a dataframe with one row

,min(CO(GT)),max(CO(GT))
0,0.1,11.9


Shown above is the simplest iteration of this method call, which evaluated across an entire column. Now, let's take a look at what happens when we start experimenting with grouping variables.

In [153]:
air.get_min_max(col = "CO(GT)", groupCol = "Date").head() #Returns a dataframe with one row per date

,Date,min(CO(GT)),max(CO(GT))
0,3/11/2004,0.6,6.9
1,3/12/2004,0.6,6.6
2,3/10/2004,1.2,2.6
3,3/13/2004,1.0,4.2
4,3/15/2004,1.0,8.1


Above is the minimum and maximum for a given date. We see that we have one minimum and one maximum value per row, which is an intuitive format for reading this data. Now let's see what happens if we forget (or decide) to leave out the column that we want to evaluate.

In [154]:
air.get_min_max() #Returns a minimum and maximum for every numeric column

,min(Unnamed: 0),min(CO(GT)),min(PT08S1(CO)),min(NMHC(GT)),min(C6H6(GT)),min(PT08S2(NMHC)),min(NOx(GT)),min(PT08S3(NOx)),min(NO2(GT)),min(PT08S4(NO2)),...,max(C6H6(GT)),max(PT08S2(NMHC)),max(NOx(GT)),max(PT08S3(NOx)),max(NO2(GT)),max(PT08S4(NO2)),max(PT08S5(O3)),max(T),max(RH),max(AH)
0,0,0.1,647,7,0.1,383,2,322,2,551,...,63.7,2214,1479,2683,340,2775,2523,44.6,88.7,2.231


Looks like we get a minimum and maximum for every numeric column in the DataFrame! We had several options for what we could do here, but choosing this behavior makes sense as the user still gets an evaluation and can use it to inform future method calls or potential edits to their program. Let's see what happens if we specify a grouping variable, but no column to evaluate.

In [155]:
air.get_min_max(groupCol = "Date").head() #Returns a minimum and maximum for every numeric column with one row per date

,Date,min(Unnamed: 0),min(CO(GT)),min(PT08S1(CO)),min(NMHC(GT)),min(C6H6(GT)),min(PT08S2(NMHC)),min(NOx(GT)),min(PT08S3(NOx)),min(NO2(GT)),...,max(C6H6(GT)),max(PT08S2(NMHC)),max(NOx(GT)),max(PT08S3(NOx)),max(NO2(GT)),max(PT08S4(NO2)),max(PT08S5(O3)),max(T),max(RH),max(AH)
0,3/11/2004,6,0.6,913.0,8.0,1.1,512.0,16.0,702.0,28.0,...,27.4,1488.0,383.0,1918.0,172.0,2333.0,1704.0,11.3,81.1,0.8778
1,3/12/2004,30,0.6,831.0,7.0,1.0,501.0,21.0,624.0,32.0,...,32.6,1610.0,340.0,1895.0,170.0,2390.0,1887.0,16.9,65.9,0.7771
2,3/10/2004,0,1.2,1197.0,38.0,4.7,750.0,89.0,1056.0,92.0,...,11.9,1046.0,172.0,1337.0,122.0,1692.0,1268.0,13.6,60.0,0.7888
3,3/13/2004,54,1.0,978.0,27.0,2.6,625.0,53.0,754.0,60.0,...,19.6,1286.0,296.0,1420.0,165.0,1922.0,1886.0,19.4,71.9,0.8193
4,3/15/2004,102,1.0,1075.0,39.0,3.9,703.0,66.0,537.0,71.0,...,39.2,1754.0,478.0,1156.0,187.0,2679.0,2184.0,24.4,70.5,1.0945


It behaves similarly to the previous two calls, where there is one row per date, and all the numeric columns are evaluated. Once again, this is consistent with previous behaviors we have demonstrated. Lastly, let's specify a non-numeric column for the evaluation and see what happens.

In [156]:
air.get_min_max(col = "Date") #Returns an error due to incorrect data type

Please provide a numeric column.


DataFrame[Unnamed: 0: bigint, Date: string, Time: string, CO(GT): double, PT08S1(CO): bigint, NMHC(GT): bigint, C6H6(GT): double, PT08S2(NMHC): bigint, NOx(GT): bigint, PT08S3(NOx): bigint, NO2(GT): bigint, PT08S4(NO2): bigint, PT08S5(O3): bigint, T: double, RH: double, AH: double]

We've returned a string message asking the user to provide a numeric column. With this, it looks like we've accounted for most of the user behavior that we expect with this function!

The last method that was created for `SparkDataCheck` is the `get_count()` function. This function essentially generates a 1-or-2-way frequency table using user-specified inputs for both variables. The important caveat with this function is that we need both variables to be string-based data types. Let's take a closer look:

In [157]:
#Test get_count
air.get_count(col = "CO(GT)") #Returns error due to non-string data type
air.get_count(col = "Date", groupCol = "CO(GT)") #Returns an error due to non-string data type

Please provide a string column.
Please provide a string column for the grouping variable.


DataFrame[Unnamed: 0: bigint, Date: string, Time: string, CO(GT): double, PT08S1(CO): bigint, NMHC(GT): bigint, C6H6(GT): double, PT08S2(NMHC): bigint, NOx(GT): bigint, PT08S3(NOx): bigint, NO2(GT): bigint, PT08S4(NO2): bigint, PT08S5(O3): bigint, T: double, RH: double, AH: double]

We've created two defined error behaviors, one for if the first column is non-string-based, and another for the grouping column. This shows that a message will return if either column doesn't meet the requirements, which is expected.

In [158]:
air.get_count(col = "Date").head() #Gets number of observations per date

,Date,count(Date)
0,3/11/2004,24
1,3/12/2004,24
2,3/10/2004,6
3,3/13/2004,24
4,3/15/2004,24


Above is a demonstration of a 1-way frequency table, counting observations by date. Now, let's look at the 2-way table:

In [159]:
air.get_count(col = "Date", groupCol = "Time").head() #Gets number of observations per date/time combo

,Date,Time,count(Date)
0,3/12/2004,11:00:00,1
1,3/11/2004,21:00:00,1
2,3/13/2004,0:00:00,1
3,3/12/2004,7:00:00,1
4,3/11/2004,18:00:00,1


While the practicality of this particular method call is questionable (since each date/time combination should be unique in this particular context), we see that it did generate a two-way count of all the observations. This means our function is working as intended! With this, we've now created a class and verified a set of expected behaviors that are possible with the methods we've created. As such, the `SparkDataCheck` class is ready for some applied usage!